# 0. Problem
## 1280. Students and Examinations — Easy
Return every student × subject combination and the number of attended examinations, including zero counts.
Official: https://leetcode.com/problems/students-and-examinations/

# 1. Setup

In [ ]:
import pandas as pd
students_rows=[(1,"Alice"),(2,"Bob"),(13,"John"),(6,"Alex")]; subjects_rows=[("Math",),("Physics",),("Programming",)]; exams_rows=[(1,"Math"),(1,"Physics"),(1,"Programming"),(2,"Programming"),(1,"Physics"),(1,"Math"),(13,"Math"),(13,"Programming"),(13,"Physics"),(2,"Math"),(1,"Math")]
students_pd=pd.DataFrame(students_rows,columns=["student_id","student_name"]); subjects_pd=pd.DataFrame(subjects_rows,columns=["subject_name"]); exams_pd=pd.DataFrame(exams_rows,columns=["student_id","subject_name"])

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
spark=SparkSession.builder.getOrCreate(); students_spark=spark.createDataFrame(students_rows,["student_id","student_name"]); subjects_spark=spark.createDataFrame(subjects_rows,["subject_name"]); exams_spark=spark.createDataFrame(exams_rows,["student_id","subject_name"])
students_spark.createOrReplaceTempView("Students"); subjects_spark.createOrReplaceTempView("Subjects"); exams_spark.createOrReplaceTempView("Examinations")

# 2. SQL Solution

In [ ]:
sql_result=spark.sql("""WITH exam_counts AS (SELECT student_id,subject_name,COUNT(*) AS attended_exams FROM Examinations GROUP BY student_id,subject_name) SELECT s.student_id,s.student_name,sub.subject_name,COALESCE(e.attended_exams,0) AS attended_exams FROM Students s CROSS JOIN Subjects sub LEFT JOIN exam_counts e ON s.student_id=e.student_id AND sub.subject_name=e.subject_name ORDER BY s.student_id,sub.subject_name"""); sql_result.show(truncate=False)

# 3. pandas Solution

In [ ]:
grid_pd=students_pd.merge(subjects_pd,how="cross"); counts_pd=exams_pd.groupby(["student_id","subject_name"],as_index=False).size().rename(columns={"size":"attended_exams"}); result_pd=grid_pd.merge(counts_pd,on=["student_id","subject_name"],how="left").fillna({"attended_exams":0}); result_pd["attended_exams"]=result_pd["attended_exams"].astype(int); result_pd=result_pd.sort_values(["student_id","subject_name"]).reset_index(drop=True); result_pd

# 4. PySpark Solution

In [ ]:
grid=students_spark.crossJoin(subjects_spark); counts=exams_spark.groupBy("student_id","subject_name").agg(F.count("*").alias("attended_exams")); result_spark=(grid.join(counts,on=["student_id","subject_name"],how="left").fillna({"attended_exams":0}).orderBy("student_id","subject_name")); result_spark.show(truncate=False)

# 5. Pattern Mapping
| Concept | SQL | pandas | PySpark |
|---|---|---|---|
| all combinations | `CROSS JOIN` | `merge(how="cross")` | `.crossJoin()` |
| keep zero counts | `LEFT JOIN + COALESCE` | left merge + `.fillna()` | left join + `.fillna()` |

# 6. Muscle-Memory Round

พิมพ์ใหม่เองโดยไม่ copy คำตอบด้านบน

In [ ]:
# MUSCLE MEMORY — SQL
# Rebuild using temp view(s): Students, Subjects, Examinations

In [ ]:
# MUSCLE MEMORY — PANDAS
# Rebuild using: students_pd, subjects_pd, exams_pd

In [ ]:
# MUSCLE MEMORY — PYSPARK
# Rebuild using: students_spark, subjects_spark, exams_spark